# A short premium portfolio

Chapters 7 and 8 of *The Unlucky Investor's Guide to Options Trading* build a portfolio of short premium: a core of strangles, a supplemental book of iron condors, laddered on staggered expiries, each sized by its probability of profit (Equation 8.1). This notebook runs that portfolio (`premium_portfolio.py`) on BTC options and compares it with a single 16 delta strangle at 45 DTE (`premium_management.py`) over the same period, both under the portfolio rules of the book: the allocation by IV rank and the caps on the BPR of one combo.

The portfolio holds up to 3 combos per book. The BPR of each is its book's share of the allocation, divided by 3; the core book has the larger share, because its probability of profit is higher. The single strangle sells 0.3 BTC of options, a size the rules do not refuse.

Set `SBT2_SERVER_TOKEN`, and `SBT2_SERVER_URL` if the server is not on this machine. The server needs the Deribit data of the period and a year of DVOL before it.

In [ ]:
import os

import pandas as pd

from sbt2.lab import Lab

url = os.environ.get("SBT2_SERVER_URL", "ws://127.0.0.1:8765")
lab = await Lab.connect(url, os.environ["SBT2_SERVER_TOKEN"])

In [ ]:
spec = {
    "instruments": ["BTC-OPTIONS.DERIBIT"],
    "period": ["2024-01-01", "2025-01-01"],
    "venue": "deribit_option",
    "capital": "1 BTC",
    "split": {"validation": 0.2, "test": 0.2},
    "part": "train",
    "chain": {"min_dte": 0, "max_dte": 60, "strike_band": "0.3"},
    "risk": {"short_premium": {}},
}

portfolio = await lab.run(spec, "premium_portfolio:PremiumPortfolio")
strangle = await lab.run(
    {**spec, "params": {"quantity": "0.3"}}, "premium_management:ShortStrangle"
)
pd.concat([portfolio.table(), strangle.table()], keys=["portfolio", "single strangle"])

The statistics over the trades of each run: the probability of profit (`POP`), the tail loss and the buying power reduction.

In [ ]:
pd.DataFrame(
    {
        "portfolio": await portfolio[0].trade_statistics(),
        "single strangle": await strangle[0].trade_statistics(),
    }
)

In [ ]:
await portfolio[0].tearsheet()

In [ ]:
await strangle[0].tearsheet()

In [ ]:
await lab.close()